# Stack Overflow Data Collection (Stack Exchange API)
Collects questions via the API and cleans HTML bodies with BeautifulSoup. Saves to CSV and resumes if re-run.

In [1]:
!pip install -q requests beautifulsoup4 lxml pandas

## 1. Settings
Get a free key at https://stackapps.com/apps/oauth/register (raises quota to 10,000 requests/day).

In [2]:
START_DATE = "2026-01-01"     # YYYY-MM-DD
END_DATE   = "2026-12-31"
API_KEY    = ""               # paste your Stack Apps key here
TAG        = None             # e.g. "python", or None for all tags
MAX_QUESTIONS = 500_000       # stop after this many (None = no limit)
OUT_CSV    = "questions.csv"

## 2. Imports and helpers

In [3]:
import csv, os, time
from datetime import datetime, timedelta, timezone
import requests
from bs4 import BeautifulSoup

API = "https://api.stackexchange.com/2.3"
UA = "StudentProject-SOAssistant/1.0 (university coursework)"
FIELDS = ["question_id", "title", "body_text", "code_text", "tags", "score", "view_count",
          "answer_count", "is_answered", "has_accepted_answer", "creation_date",
          "closed_reason", "duplicate_of"]


def clean_html(html):
    """Split question HTML into plain prose and code blocks."""
    if not html:
        return "", ""
    soup = BeautifulSoup(html, "lxml")
    pres = soup.find_all("pre")
    code = "\n\n".join(pre.get_text("\n", strip=True) for pre in pres)
    for pre in pres:
        pre.decompose()
    text = " ".join(soup.get_text(" ", strip=True).split())
    return text, code


def load_seen_ids(path):
    """Read IDs already saved so collection can resume."""
    if not os.path.exists(path):
        return set()
    with open(path, newline="", encoding="utf-8") as f:
        return {row["question_id"] for row in csv.DictReader(f)}


def append_rows(path, rows):
    new_file = not os.path.exists(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if new_file:
            w.writeheader()
        w.writerows(rows)


def make_filter(key):
    """Create an API filter that includes the body and duplicate links."""
    params = {"include": "question.body;question.closed_details;question.closed_reason",
              "base": "default", "unsafe": "false"}
    if key:
        params["key"] = key
    r = requests.get(f"{API}/filters/create", params=params, timeout=30)
    r.raise_for_status()
    return r.json()["items"][0]["filter"]


def parse_item(q):
    body, code = clean_html(q.get("body", ""))
    originals = (q.get("closed_details") or {}).get("original_questions") or []
    return {
        "question_id": q["question_id"],
        "title": BeautifulSoup(q.get("title", ""), "lxml").get_text(),
        "body_text": body,
        "code_text": code,
        "tags": "|".join(q.get("tags", [])),
        "score": q.get("score", 0),
        "view_count": q.get("view_count", 0),
        "answer_count": q.get("answer_count", 0),
        "is_answered": q.get("is_answered", False),
        "has_accepted_answer": "accepted_answer_id" in q,
        "creation_date": datetime.fromtimestamp(q["creation_date"], timezone.utc).isoformat(),
        "closed_reason": q.get("closed_reason", ""),
        "duplicate_of": "|".join(str(o["question_id"]) for o in originals),
    }

/Users/bipin/Documents/aberdeen msc ai/applied_artificial_intelligence/assessment_1/assessment_1_task_1/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 3. Collector

In [4]:
def collect(start, end, key, out, tag=None, max_questions=None):
    flt = make_filter(key)
    seen = load_seen_ids(out)
    total, t0 = len(seen), time.time()
    day = datetime.strptime(start, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    stop = datetime.strptime(end, "%Y-%m-%d").replace(tzinfo=timezone.utc)

    while day <= stop:                      # one day at a time keeps paging shallow
        page, has_more = 1, True
        while has_more:
            params = {"site": "stackoverflow", "pagesize": 100, "page": page,
                      "order": "asc", "sort": "creation", "filter": flt,
                      "fromdate": int(day.timestamp()),
                      "todate": int((day + timedelta(days=1)).timestamp()) - 1}
            if key:
                params["key"] = key
            if tag:
                params["tagged"] = tag

            try:
                data = requests.get(f"{API}/questions", params=params, timeout=30,
                                    headers={"User-Agent": UA}).json()
            except (requests.RequestException, ValueError) as e:
                print("Request failed, retrying in 10s:", e)
                time.sleep(10)
                continue

            if "error_id" in data:
                print("API error:", data.get("error_message"))
                if data["error_id"] == 502:  # throttled
                    time.sleep(60)
                    continue
                return

            rows = [parse_item(q) for q in data.get("items", [])
                    if str(q["question_id"]) not in seen]
            append_rows(out, rows)
            seen.update(str(r["question_id"]) for r in rows)
            total += len(rows)
            print(f"{day.date()} p{page}: +{len(rows)} (total {total}, "
                  f"quota left {data.get('quota_remaining')})")

            if data.get("quota_remaining", 1) == 0:
                print("Daily quota used up. Re-run tomorrow; collection resumes.")
                return
            if max_questions and total >= max_questions:
                print(f"Reached {total} questions in {time.time() - t0:.0f}s")
                return
            if "backoff" in data:            # API asks us to wait
                time.sleep(data["backoff"] + 1)

            has_more = data.get("has_more", False)
            page += 1
            time.sleep(0.1)
        day += timedelta(days=1)

    print(f"Done: {total} questions in {time.time() - t0:.0f}s")

## 4. Run
You can stop this cell at any time (Interrupt kernel); re-running continues from where it stopped.

In [5]:
collect(START_DATE, END_DATE, API_KEY or None, OUT_CSV, TAG, MAX_QUESTIONS)

2024-01-01 p1: +100 (total 100, quota left 298)
2024-01-01 p2: +100 (total 200, quota left 298)
2024-01-01 p3: +100 (total 300, quota left 296)
2024-01-01 p4: +100 (total 400, quota left 294)
2024-01-01 p5: +100 (total 500, quota left 294)
2024-01-01 p6: +100 (total 600, quota left 292)
2024-01-01 p7: +100 (total 700, quota left 292)
2024-01-01 p8: +94 (total 794, quota left 290)
2024-01-02 p1: +100 (total 894, quota left 289)
2024-01-02 p2: +100 (total 994, quota left 289)
2024-01-02 p3: +100 (total 1094, quota left 287)
2024-01-02 p4: +100 (total 1194, quota left 287)
2024-01-02 p5: +100 (total 1294, quota left 285)
2024-01-02 p6: +100 (total 1394, quota left 285)
2024-01-02 p7: +100 (total 1494, quota left 283)
2024-01-02 p8: +100 (total 1594, quota left 283)
2024-01-02 p9: +100 (total 1694, quota left 281)
2024-01-02 p10: +100 (total 1794, quota left 280)
2024-01-02 p11: +100 (total 1894, quota left 280)
2024-01-02 p12: +100 (total 1994, quota left 278)
2024-01-02 p13: +100 (total 

/var/folders/jx/vfzctgrd6svfxbn5yt9s35r40000gn/T/ipykernel_45958/3492192524.py:59: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a filename than HTML or XML.

If you meant to use Beautiful Soup to parse the contents of a file on disk, then something has gone wrong. You should open the file first, using code like this:

    filehandle = open(your filename)

You can then feed the open filehandle into Beautiful Soup instead of using the filename.

However, if you want to parse some data that happens to look like a filename, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  "title": BeautifulSoup(q.get("title", ""), "lxml").get_text(),


2024-01-06 p4: +100 (total 7654, quota left 221)
2024-01-06 p5: +100 (total 7754, quota left 219)
2024-01-06 p6: +100 (total 7854, quota left 219)
2024-01-06 p7: +100 (total 7954, quota left 217)
2024-01-06 p8: +100 (total 8054, quota left 217)
2024-01-06 p9: +100 (total 8154, quota left 215)
2024-01-06 p10: +36 (total 8190, quota left 214)
2024-01-07 p1: +100 (total 8290, quota left 214)
2024-01-07 p2: +100 (total 8390, quota left 212)
2024-01-07 p3: +100 (total 8490, quota left 212)
2024-01-07 p4: +100 (total 8590, quota left 210)
2024-01-07 p5: +100 (total 8690, quota left 210)
2024-01-07 p6: +100 (total 8790, quota left 208)
2024-01-07 p7: +100 (total 8890, quota left 208)
2024-01-07 p8: +100 (total 8990, quota left 206)
2024-01-07 p9: +100 (total 9090, quota left 206)
2024-01-07 p10: +66 (total 9156, quota left 204)
2024-01-08 p1: +100 (total 9256, quota left 204)
2024-01-08 p2: +100 (total 9356, quota left 202)
2024-01-08 p3: +100 (total 9456, quota left 202)
2024-01-08 p4: +100 

## 5. Quick check

In [6]:
import pandas as pd
df = pd.read_csv(OUT_CSV)
print(len(df), "questions")
print("Answered rate:", df["is_answered"].mean().round(3))
print("Duplicates:", (df["closed_reason"] == "Duplicate").sum())
df.head()

29038 questions
Answered rate: 0.535
Duplicates: 1006


,question_id,title,body_text,code_text,tags,score,view_count,answer_count,is_answered,has_accepted_answer,creation_date,closed_reason,duplicate_of
0,77740646,Make URL Address Text a Clickable URL Using se...,Happy New Year to Everyone.. I have a list of ...,function createPDFDocuments() {\n var sheetD...,google-apps-script|pdf-generation,2,741,1,True,True,2024-01-01T00:01:14+00:00,NaN,NaN
1,77740652,7zip commandline extract excluding single file...,Using 7z.exe (v19.00 x64) from the commandline...,"""C:\Program Files\7-Zip\7z"" x -bb3 -x""Www.WebA...",extract|7zip,0,553,1,True,True,2024-01-01T00:04:07+00:00,NaN,NaN
2,77740661,How do I delete an element from an array and s...,Im building a chrome extension that lets you s...,"$(""ul"").on(""click"", "".removeButton"", function(...",javascript|arrays|google-chrome-extension|goog...,-1,31,0,False,False,2024-01-01T00:08:51+00:00,Duplicate,20279484
3,77740664,Issues with JumpList in C#,I am working on a C# application and trying to...,CS0176: Member 'JumpList.AddToRecent(string)' ...,c#|jump-list,0,114,0,False,False,2024-01-01T00:13:08+00:00,NaN,NaN
4,77740666,Determine whether app launches AOT binary (or ...,I am developing an Android Test App and was wo...,$ adb shell cmd package compile -m speed -f my...,android,1,118,0,False,False,2024-01-01T00:14:37+00:00,NaN,NaN
